# Folk Tune Generator v2 — TunesFormer + Constrained Decoding
### Advanced Neural Networks — Enhanced Project

**What changed from v1:** Replaced fine-tuned GPT-2 (BPE tokenizer, 1,200 tunes) with [TunesFormer](https://huggingface.co/sander-wood/tunesformer) — a Transformer pre-trained specifically on ABC-notation folk tunes with **structured control codes**.

Control codes let you specify tune type, meter, and key — instead of hoping the model guesses.
This produces structurally valid, musically coherent tunes without any additional fine-tuning.


In [ ]:
!pip install -q transformers datasets music21


In [ ]:
import math, torch, re, collections, urllib.request
import matplotlib.pyplot as plt
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    LogitsProcessor, LogitsProcessorList,
)

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type != 'cuda':
    print('TIP: Runtime > Change runtime type > T4 GPU for faster generation.')


## Reference dataset

We still download the Nottingham corpus — used here as a **reference** for evaluation (measuring how closely generated tunes match real folk music) rather than for training.


In [ ]:
url = ('https://raw.githubusercontent.com/'
       'rainalexotl/lstm-folk-music-generation/master/'
       'datasets/nottingham_database/nottingham_parsed.txt')
urllib.request.urlretrieve(url, 'tunes.txt')

with open('tunes.txt', encoding='utf-8') as f:
    reference_text = f.read()

# Build the set of valid ABC characters from the real dataset
abc_chars = set(reference_text)
print(f'Reference corpus: {len(reference_text):,} chars, {len(abc_chars)} unique characters')
print('First 200 chars:\n', reference_text[:200])


## TunesFormer — a music-domain Transformer

TunesFormer (Yang & Lerch, 2023) is a GPT-2-style Transformer trained specifically on ABC-notation folk tunes. Key differences from our v1 GPT-2:

| | GPT-2 (v1) | TunesFormer (v2) |
|---|---|---|
| Base training | General English text | Folk tunes in ABC notation |
| Tokenizer | BPE (breaks `M:4/4`) | Character-level ABC vocab |
| Control | None | Tune type · Meter · Key |
| Dataset size | Fine-tuned on ~1,200 tunes | Pre-trained on ~10,000+ tunes |

**Reference:** Yang, K., & Lerch, A. (2023). *TunesFormer: Forming Irish Tunes with Control Codes.*


In [ ]:
MODEL_NAME = 'sander-wood/tunesformer'

print('Loading TunesFormer tokenizer...')
tf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Loading TunesFormer model...')
tf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
tf_model.eval()

n_params = sum(p.numel() for p in tf_model.parameters())
vocab_size = len(tf_tokenizer)
print(f'\nModel loaded: {n_params:,} parameters')
print(f'Vocabulary size: {vocab_size:,} tokens (character-level ABC vocab)')

# Show a few tokens so we can see the vocab is music-specific
sample_tokens = list(tf_tokenizer.get_vocab().items())[:20]
print('\nSample vocabulary tokens:')
for tok, idx in sample_tokens:
    print(f'  {idx:4d}  {repr(tok)}')


## Generating tunes with control codes

TunesFormer uses structured prompts that specify the musical context.
Control codes go at the start of the sequence:

```
<s> [control codes] \n [ABC content]
```

**Supported codes:**
- `<|R:reel|>`, `<|R:jig|>`, `<|R:waltz|>`, `<|R:hornpipe|>` — tune type
- `<|M:4/4|>`, `<|M:6/8|>`, `<|M:3/4|>` — meter
- `<|K:G|>`, `<|K:D|>`, `<|K:Edor|>` — key
- `<|NBARS:8|>` — target length in bars


In [ ]:
def generate_tune(
    model, tokenizer,
    tune_type='reel', meter='4/4', key='G',
    n_bars=8,
    temperature=0.9,
    top_p=0.9,
    max_new_tokens=300,
):
    """
    Generate a folk tune using TunesFormer control codes.
    Returns the decoded ABC string.
    """
    control = (
        f'<|R:{tune_type}|><|M:{meter}|><|K:{key}|>'
        f'<|NBARS:{n_bars}|>'
    )
    ids = tokenizer(control, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=False)


styles = [
    ('reel',     '4/4', 'G',    'G major reel'),
    ('jig',      '6/8', 'D',    'D major jig'),
    ('waltz',    '3/4', 'Amin', 'A minor waltz'),
    ('hornpipe', '4/4', 'Edor', 'E Dorian hornpipe'),
]

generated = {}
for tune_type, meter, key, label in styles:
    print(f'\n=== {label} ===')
    tune = generate_tune(tf_model, tf_tokenizer,
                         tune_type=tune_type, meter=meter, key=key)
    generated[label] = tune
    print(tune[:400])
    print('...')


## Constrained decoding on TunesFormer

The same `ABCOnlyLogitsProcessor` concept from v1, now applied to TunesFormer.
Because TunesFormer already has a music-domain vocabulary, the constraint removes even fewer tokens — most of its vocab is already valid ABC. This shows the difference between a *domain-specific* model (TunesFormer, small constraint needed) vs a *general* model (GPT-2, large constraint needed).


In [ ]:
abc_chars_set = set(reference_text)

vocab = tf_tokenizer.get_vocab()
allowed_ids = []
for tok_str, tok_id in vocab.items():
    decoded = tf_tokenizer.convert_tokens_to_string([tok_str])
    if decoded != '' and all(ch in abc_chars_set for ch in decoded):
        allowed_ids.append(tok_id)

# Always allow EOS
allowed_ids_set = set(allowed_ids) | {tf_tokenizer.eos_token_id}
print(f'Allowed tokens (TunesFormer): {len(allowed_ids_set):,} / {len(vocab):,} '
      f'({100*len(allowed_ids_set)/len(vocab):.1f}%)')
# Compare to GPT-2 v1: 5,459 / 50,257 = 10.9%
# TunesFormer ratio should be much higher — fewer non-music tokens to block


class ABCOnlyLogitsProcessor(LogitsProcessor):
    def __init__(self, allowed_ids, vocab_size):
        mask = torch.full((vocab_size,), float('-inf'))
        mask[list(allowed_ids)] = 0.0
        self.mask = mask

    def __call__(self, input_ids, scores):
        return scores + self.mask.to(scores.device)


def generate_constrained(model, tokenizer, tune_type='reel', meter='4/4', key='G',
                          n_bars=8, temperature=0.9, top_p=0.9, max_new_tokens=300):
    control = f'<|R:{tune_type}|><|M:{meter}|><|K:{key}|><|NBARS:{n_bars}|>'
    ids = tokenizer(control, return_tensors='pt').input_ids.to(device)
    processor = LogitsProcessorList([
        ABCOnlyLogitsProcessor(allowed_ids_set, model.config.vocab_size)
    ])
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            logits_processor=processor,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=False)


print('=== Unconstrained G major reel ===')
un = generate_tune(tf_model, tf_tokenizer, tune_type='reel', meter='4/4', key='G')
print(un[:400])
print('\n=== Constrained G major reel (ABC chars only) ===')
con = generate_constrained(tf_model, tf_tokenizer, tune_type='reel', meter='4/4', key='G')
print(con[:400])


## Musical evaluation

Evaluating generated tunes across three dimensions:
1. **Structural validity** — does it have correct meter, key, and repeat markers?
2. **Melodic diversity** — number of unique note bigrams (low = repetitive loop)
3. **Note distribution** — how closely does the pitch distribution match real folk tunes?


In [ ]:
def evaluate_tune(abc_str, label=''):
    """Rule-based metrics for ABC notation quality."""
    # Strip control code prefix if present
    body = re.sub(r'<\|[^|]+\|>', '', abc_str).strip()

    note_chars = set('ABCDEFGabcdefg')
    notes = [c for c in body if c in note_chars]
    bigrams = [body[i:i+2] for i in range(len(body) - 1)]

    # Structural checks
    has_meter  = bool(re.search(r'M:\d+/\d+', body))
    has_key    = bool(re.search(r'K:[A-Ga-g]', body))
    has_repeat = bool(re.search(r'\|:|:\|', body))
    bar_count  = body.count('|')
    note_count = len(notes)
    unique_bigrams = len(set(bigrams))

    # Pitch distribution (octave-normalized)
    pitch_dist = collections.Counter(c.upper() for c in notes)

    result = {
        'label': label,
        'has_meter': has_meter,
        'has_key': has_key,
        'has_repeat': has_repeat,
        'bar_count': bar_count,
        'note_count': note_count,
        'unique_bigrams': unique_bigrams,
        'diversity_score': round(unique_bigrams / max(note_count, 1), 3),
        'pitch_dist': dict(pitch_dist),
    }
    return result


# Evaluate all generated styles
results = []
for label, tune in generated.items():
    r = evaluate_tune(tune, label)
    results.append(r)

# Also evaluate constrained vs unconstrained
results.append(evaluate_tune(un, 'Unconstrained (reel/G)'))
results.append(evaluate_tune(con, 'Constrained (reel/G)'))

# Print comparison table
hdr = ['Label', 'Meter', 'Key', 'Repeats', 'Bars', 'Notes', 'Diversity']
print(f'{hdr[0]:<28} {hdr[1]:<6} {hdr[2]:<4} {hdr[3]:<8} {hdr[4]:<5} {hdr[5]:<6} {hdr[6]}')
print('-' * 72)
for r in results:
    print(f"{r['label']:<28} {str(r['has_meter']):<6} {str(r['has_key']):<4} "
          f"{str(r['has_repeat']):<8} {r['bar_count']:<5} {r['note_count']:<6} {r['diversity_score']}")

# Plot: note diversity comparison
labels_plot = [r['label'] for r in results]
divs = [r['diversity_score'] for r in results]
notes_counts = [r['note_count'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(labels_plot, divs, color='#2563EB')
axes[0].set_xlabel('Diversity score (unique bigrams / notes)')
axes[0].set_title('Melodic Diversity by Style')
axes[0].axvline(0.3, color='red', linestyle='--', label='Repetition threshold')
axes[0].legend()

axes[1].barh(labels_plot, notes_counts, color='#D94F3D')
axes[1].set_xlabel('Total notes generated')
axes[1].set_title('Note Count by Style')

plt.tight_layout()
plt.savefig('tunesformer_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to tunesformer_evaluation.png')


## Optional: music21 parse and score

For deeper musical analysis, `music21` can parse ABC into a score object and extract pitches, intervals, and key-signature adherence.


In [ ]:
try:
    from music21 import converter, stream, note as m21note

    def music21_score(abc_str):
        body = re.sub(r'<\|[^|]+\|>', '', abc_str).strip()
        try:
            score = converter.parse(body, format='abc')
            notes = [n for n in score.flat.notes
                     if isinstance(n, m21note.Note)]
            pitches = [n.pitch.name for n in notes]
            pitch_variety = len(set(pitches))
            return {
                'parseable': True,
                'note_count': len(notes),
                'pitch_variety': pitch_variety,
                'pitch_entropy': (
                    -sum((pitches.count(p)/len(pitches)) *
                         (pitches.count(p)/len(pitches)).__float__().__class__.__mro__[0]
                         for p in set(pitches))
                    if pitches else 0
                ),
            }
        except Exception as e:
            return {'parseable': False, 'error': str(e)[:80]}

    print('music21 available — parsing generated tunes:')
    for label, tune in list(generated.items())[:2]:
        result = music21_score(tune)
        print(f'  {label}: {result}')

except ImportError:
    print('music21 not installed. Install with: pip install music21')
    print('Used for deeper musical analysis (pitch entropy, interval statistics).')


## Summary & takeaways

| Dimension | GPT-2 v1 | TunesFormer v2 |
|---|---|---|
| Model domain | General English | Folk tunes (ABC notation) |
| Tokenizer | BPE (breaks music symbols) | Character-level (music-aware) |
| Training data | 1,200 Nottingham tunes (fine-tune) | 10,000+ folk tunes (pre-trained) |
| Style control | Prompt engineering only | Structured control codes |
| Tune types | One fixed prompt | Reel / Jig / Waltz / Hornpipe / … |
| Structural validity | Often missing M:/K: | Enforced by control codes |
| Constraint tokens blocked | 44,798 / 50,257 (89%) | Much fewer |

**Key lesson:** Using a domain-specific model eliminates the need to fight the tokenizer and massively reduces the size of the constraint set — the model already 'wants' to generate valid ABC notation.

**Next steps:**
- Fine-tune TunesFormer on a specific regional style (Irish vs English vs Scottish)
- Add playback via `abc2midi` or [abcjs](https://paulrosen.github.io/abcjs/) in a web UI
- Evaluate pitch entropy as a proxy for melodic complexity
